<a href="https://colab.research.google.com/github/Ganasa18/belajar-tensorflow/blob/main/lab_action_classifier_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =========================================================
# CELL 1 — GOOGLE DRIVE + TRAINING PATH SETUP
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

from google.colab import drive

import os


# =========================================================
# 1. MOUNT GOOGLE DRIVE
# =========================================================

DRIVE_MOUNT = "/content/drive"

drive.mount(
    DRIVE_MOUNT,
    force_remount=False
)

assert os.path.exists(
    f"{DRIVE_MOUNT}/MyDrive"
), "Google Drive belum mounted"


# =========================================================
# 2. STORAGE DIRECTORY
# =========================================================

SAVE_DIR = (
    f"{DRIVE_MOUNT}/MyDrive/action_classifier"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)


# =========================================================
# 3. SUBDIRECTORIES
# =========================================================

RAW_DIR = (
    f"{SAVE_DIR}/raw"
)

PROCESSED_DIR = (
    f"{SAVE_DIR}/processed"
)

TEACHER_DIR = (
    f"{SAVE_DIR}/teacher"
)

TRUSTED_DIR = (
    f"{SAVE_DIR}/trusted"
)

REPAIR_DIR = (
    f"{SAVE_DIR}/repair"
)

BLIND_TEST_DIR = (
    f"{SAVE_DIR}/blind_test"
)

MODEL_DIR = (
    f"{SAVE_DIR}/models"
)

ONNX_DIR = (
    f"{MODEL_DIR}/onnx"
)

REPORT_DIR = (
    f"{SAVE_DIR}/reports"
)


for directory in [
    RAW_DIR,
    PROCESSED_DIR,
    TEACHER_DIR,
    TRUSTED_DIR,
    REPAIR_DIR,
    BLIND_TEST_DIR,
    MODEL_DIR,
    ONNX_DIR,
    REPORT_DIR,
]:

    os.makedirs(
        directory,
        exist_ok=True
    )


# =========================================================
# 4. DATASET PATHS
# =========================================================

MANUAL_SEED_PATH = (
    f"{RAW_DIR}/manual_seed_v1.jsonl"
)

PUBLIC_RAW_PATH = (
    f"{RAW_DIR}/public_seed_raw.jsonl"
)

NORMALIZED_DATASET_PATH = (
    f"{PROCESSED_DIR}/normalized_dataset_v1.jsonl"
)

DEDUP_DATASET_PATH = (
    f"{PROCESSED_DIR}/dedup_dataset_v1.jsonl"
)

TEACHER_LABELED_PATH = (
    f"{TEACHER_DIR}/teacher_labeled_v1.jsonl"
)

TEACHER_REJECTED_PATH = (
    f"{TEACHER_DIR}/teacher_rejected_v1.jsonl"
)

TRUSTED_TRAIN_PATH = (
    f"{TRUSTED_DIR}/action_classifier_trusted_v1.jsonl"
)

REPAIR_DATASET_PATH = (
    f"{REPAIR_DIR}/targeted_repair_v1.jsonl"
)

BLIND_TEST_PATH = (
    f"{BLIND_TEST_DIR}/blind_test_v1.jsonl"
)


# =========================================================
# 5. MODEL OUTPUT PATHS
# =========================================================

MODEL_PATH = (
    f"{MODEL_DIR}/action_classifier.joblib"
)

BEST_MODEL_PATH = (
    f"{MODEL_DIR}/action_classifier_best.joblib"
)

MLB_PATH = (
    f"{MODEL_DIR}/multilabel_binarizer.joblib"
)

ONNX_MODEL_PATH = (
    f"{ONNX_DIR}/action_classifier.onnx"
)


# =========================================================
# 6. REPORT PATHS
# =========================================================

METRICS_PATH = (
    f"{REPORT_DIR}/training_metrics.json"
)

PREDICTIONS_PATH = (
    f"{REPORT_DIR}/test_predictions.csv"
)

ERROR_ANALYSIS_PATH = (
    f"{REPORT_DIR}/error_analysis.csv"
)

LABEL_DISTRIBUTION_PATH = (
    f"{REPORT_DIR}/label_distribution.csv"
)

ONNX_PARITY_PATH = (
    f"{REPORT_DIR}/onnx_parity.json"
)

BENCHMARK_PATH = (
    f"{REPORT_DIR}/latency_benchmark.json"
)


# =========================================================
# 7. STATUS
# =========================================================

print()
print("=" * 70)
print("MODEL 01 — ACTION CLASSIFIER STORAGE")
print("=" * 70)

print(
    "SAVE_DIR       :",
    SAVE_DIR
)

print(
    "Manual seed    :",
    MANUAL_SEED_PATH
)

print(
    "Trusted train  :",
    TRUSTED_TRAIN_PATH
)

print(
    "Repair dataset :",
    REPAIR_DATASET_PATH
)

print(
    "Model output   :",
    MODEL_PATH
)

print(
    "Best model     :",
    BEST_MODEL_PATH
)

print(
    "ONNX model     :",
    ONNX_MODEL_PATH
)

print(
    "Metrics        :",
    METRICS_PATH
)

print("=" * 70)

Mounted at /content/drive

MODEL 01 — ACTION CLASSIFIER STORAGE
SAVE_DIR       : /content/drive/MyDrive/action_classifier
Manual seed    : /content/drive/MyDrive/action_classifier/raw/manual_seed_v1.jsonl
Trusted train  : /content/drive/MyDrive/action_classifier/trusted/action_classifier_trusted_v1.jsonl
Repair dataset : /content/drive/MyDrive/action_classifier/repair/targeted_repair_v1.jsonl
Model output   : /content/drive/MyDrive/action_classifier/models/action_classifier.joblib
Best model     : /content/drive/MyDrive/action_classifier/models/action_classifier_best.joblib
ONNX model     : /content/drive/MyDrive/action_classifier/models/onnx/action_classifier.onnx
Metrics        : /content/drive/MyDrive/action_classifier/reports/training_metrics.json


In [2]:
# =========================================================
# CELL 2 — DEPENDENCIES + IMPORTS
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

!pip -q install \
    pandas \
    numpy \
    scikit-learn \
    datasets \
    huggingface_hub \
    tqdm \
    matplotlib \
    joblib \
    requests \
    skl2onnx \
    onnx \
    onnxruntime

print("Dependencies installed.")


# =========================================================
# IMPORTS
# =========================================================

import os
import re
import json
import time
import random
import hashlib
import warnings

import requests
import joblib

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

from sklearn.model_selection import (
    train_test_split
)

from sklearn.preprocessing import (
    MultiLabelBinarizer
)

from sklearn.metrics import (
    classification_report,
    f1_score,
    accuracy_score,
    hamming_loss,
)


# =========================================================
# RANDOM SEED
# =========================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

warnings.filterwarnings(
    "ignore"
)


# =========================================================
# STATUS
# =========================================================

print()
print("=" * 70)
print("ACTION CLASSIFIER — ENVIRONMENT")
print("=" * 70)

print(
    "Random seed :",
    SEED
)

print(
    "NumPy       :",
    np.__version__
)

print(
    "Pandas      :",
    pd.__version__
)

print("=" * 70)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.2/317.2 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 68.9 MB/s eta 0:00:00
Dependencies installed.

ACTION CLASSIFIER — ENVIRONMENT
Random seed : 42
NumPy       : 2.1.3
Pandas      : 2.2.3


In [3]:
# =========================================================
# CELL 3 — ACTION TAXONOMY + DATASET SCHEMA
# MODEL 01 — ACTION CLASSIFIER
# =========================================================


# =========================================================
# 1. ACTION LABELS
# =========================================================

LABELS = [
    "READ",
    "WRITE",
    "DELETE",
    "EXECUTE",
    "NETWORK",
    "INSTALL",
    "PRIVILEGED",
    "SYSTEM_CHANGE",
]


LABEL_TO_ID = {
    label: index
    for index, label in enumerate(LABELS)
}

ID_TO_LABEL = {
    index: label
    for label, index in LABEL_TO_ID.items()
}


# =========================================================
# 2. DATASET COLUMNS
# =========================================================

DATASET_COLUMNS = [
    "command",
    "description",
    "actions",
    "confidence",
    "ambiguous",
    "source",
    "split_origin",
]


# =========================================================
# 3. VALIDATION FUNCTION
# =========================================================

def validate_actions(actions):

    if not isinstance(
        actions,
        list
    ):
        return False

    if len(actions) == 0:
        return False

    if len(actions) != len(set(actions)):
        return False

    for action in actions:

        if action not in LABELS:
            return False

    return True


# =========================================================
# 4. SCHEMA
# =========================================================

SCHEMA = {

    "model": (
        "action_classifier"
    ),

    "version": (
        "v1"
    ),

    "task": (
        "multi_label_classification"
    ),

    "labels": (
        LABELS
    ),

    "columns": (
        DATASET_COLUMNS
    ),
}


SCHEMA_PATH = (
    f"{PROCESSED_DIR}/dataset_schema_v1.json"
)


with open(
    SCHEMA_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        SCHEMA,
        f,
        indent=2,
        ensure_ascii=False
    )


# =========================================================
# 5. STATUS
# =========================================================

print()
print("=" * 70)
print("ACTION CLASSIFIER — TAXONOMY")
print("=" * 70)

for index, label in enumerate(LABELS):

    print(
        f"{index:2} -> {label}"
    )


print()
print(
    "Total labels :",
    len(LABELS)
)

print(
    "Task         :",
    "MULTI-LABEL"
)

print(
    "Schema       :",
    SCHEMA_PATH
)

print("=" * 70)


ACTION CLASSIFIER — TAXONOMY
 0 -> READ
 1 -> WRITE
 2 -> DELETE
 3 -> EXECUTE
 4 -> NETWORK
 5 -> INSTALL
 6 -> PRIVILEGED
 7 -> SYSTEM_CHANGE

Total labels : 8
Task         : MULTI-LABEL
Schema       : /content/drive/MyDrive/action_classifier/processed/dataset_schema_v1.json


In [4]:
# =========================================================
# CELL 4 — TEACHER API SETUP
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

import requests
import time
import re
import json

from google.colab import userdata


# =========================================================
# API CONFIG
# =========================================================

BASE_URL = (
    "https://api.deepseek.com/chat/completions"
)

API_KEY = userdata.get(
    "DEEPSEEK"
)

MODEL = "deepseek-v4-flash"

TEMPERATURE = 0

TIMEOUT_CONNECT = 10
TIMEOUT_READ = 45


if not API_KEY:

    raise RuntimeError(
        "DEEPSEEK API key tidak ditemukan "
        "di Colab Secrets."
    )


# =========================================================
# TEACHER CLASSIFICATION PROMPT
# =========================================================

TEACHER_SYSTEM_PROMPT = """
You are a strict shell-command action classifier.

Your task is NOT to execute commands.

Your task is NOT to judge whether a command is malicious.

Your task is ONLY to identify what ACTIONS the command
would perform if executed.

The classification is MULTI-LABEL.

A command may have one or multiple action labels.

Allowed labels:

READ
WRITE
DELETE
EXECUTE
NETWORK
INSTALL
PRIVILEGED
SYSTEM_CHANGE


=========================================================
READ
=========================================================

The command reads, displays, lists, inspects, queries,
or retrieves local information without intentionally
modifying it.

Examples:

cat /etc/os-release
-> READ

ls -la /tmp
-> READ

systemctl status ssh
-> READ


=========================================================
WRITE
=========================================================

The command creates, copies, moves, overwrites,
appends, downloads, or otherwise writes data
to local storage.

Examples:

echo hello > output.txt
-> WRITE

cp source.txt backup.txt
-> READ + WRITE

curl <TEST_URL> -o file
-> NETWORK + WRITE


=========================================================
DELETE
=========================================================

The command removes files, directories, records,
or other persistent data.

Examples:

rm test.txt
-> DELETE

rm -r <TEMP_DIR>
-> DELETE


=========================================================
EXECUTE
=========================================================

The command launches, runs, evaluates, invokes,
or restarts executable code, programs, scripts,
commands, or services.

Examples:

python app.py
-> EXECUTE

bash script.sh
-> EXECUTE

sudo systemctl restart nginx
-> EXECUTE + PRIVILEGED + SYSTEM_CHANGE


=========================================================
NETWORK
=========================================================

The command sends, receives, downloads, uploads,
queries, connects to, scans, or otherwise interacts
with a network or remote host.

Examples:

curl <TEST_URL>
-> NETWORK

ping <LAB_HOST>
-> NETWORK

wget <TEST_URL> -O file
-> NETWORK + WRITE


=========================================================
INSTALL
=========================================================

The command installs, adds, upgrades, or removes
software packages or software dependencies.

Examples:

pip install requests
-> NETWORK + INSTALL + SYSTEM_CHANGE

sudo apt install nginx
-> NETWORK + INSTALL + PRIVILEGED + SYSTEM_CHANGE


=========================================================
PRIVILEGED
=========================================================

The command explicitly requests elevated privileges
or performs an operation requiring an elevated
privilege boundary.

Examples:

sudo apt update
-> NETWORK + PRIVILEGED + SYSTEM_CHANGE

sudo systemctl restart ssh
-> EXECUTE + PRIVILEGED + SYSTEM_CHANGE


=========================================================
SYSTEM_CHANGE
=========================================================

The command modifies system state, configuration,
packages, services, permissions, users, system files,
mounts, firewall state, or other operating-system
configuration.

Examples:

chmod 600 file
-> SYSTEM_CHANGE

sudo systemctl restart nginx
-> EXECUTE + PRIVILEGED + SYSTEM_CHANGE

sudo apt install nginx
-> NETWORK + INSTALL + PRIVILEGED + SYSTEM_CHANGE


=========================================================
IMPORTANT RULES
=========================================================

1. Return every action clearly implied by the command.

2. Do NOT classify based on whether the command is
   good, bad, suspicious, malicious, or dangerous.

3. Risk classification belongs to another model.

4. Focus only on observable command behavior.

5. Do not invent actions that are not implied.

6. If the command is genuinely unclear or cannot be
   reliably classified, set:

   "ambiguous": true

7. Confidence must reflect classification certainty.

8. Shell chaining must be classified across the
   entire command.

Example:

cat input.txt | curl -X POST <TEST_URL> -d @-

-> READ + NETWORK


=========================================================
OUTPUT
=========================================================

Return ONLY valid JSON.

Schema:

{
  "actions": ["LABEL"],
  "confidence": 0.95,
  "ambiguous": false
}

actions:
- must be a JSON array
- may contain one or multiple allowed labels
- must not contain duplicates

confidence:
- number from 0 to 1

ambiguous:
- true or false

Do not include explanations.
Do not include markdown.
Do not include reasoning.
"""


# =========================================================
# TEACHER CALL
# =========================================================

def call_teacher(command):

    headers = {

        "Authorization":
            f"Bearer {API_KEY}",

        "Content-Type":
            "application/json",
    }


    payload = {

        "model":
            MODEL,

        "temperature":
            TEMPERATURE,

        "thinking": {
            "type": "disabled"
        },

        "response_format": {
            "type": "json_object"
        },

        "messages": [

            {
                "role": "system",
                "content": TEACHER_SYSTEM_PROMPT,
            },

            {
                "role": "user",
                "content": command,
            },
        ],
    }


    start = time.time()


    response = requests.post(

        BASE_URL,

        headers=headers,

        json=payload,

        timeout=(
            TIMEOUT_CONNECT,
            TIMEOUT_READ
        ),
    )


    latency = (
        time.time()
        - start
    )


    response.raise_for_status()


    data = response.json()


    content = (
        data["choices"][0]
            ["message"]
            ["content"]
    )


    return content, latency


# =========================================================
# PARSER
# =========================================================

def parse_teacher_output(text):

    if not text:

        raise ValueError(
            "Teacher response kosong."
        )


    text = text.strip()


    text = re.sub(
        r"^```(?:json)?\s*",
        "",
        text,
        flags=re.I
    )


    text = re.sub(
        r"\s*```$",
        "",
        text
    )


    match = re.search(
        r"\{[\s\S]*?\}",
        text
    )


    if not match:

        raise ValueError(
            "JSON tidak ditemukan: "
            + text[:300]
        )


    data = json.loads(
        match.group()
    )


    actions = data.get(
        "actions",
        []
    )


    confidence = float(
        data.get(
            "confidence"
        )
    )


    ambiguous = data.get(
        "ambiguous"
    )


    # =====================================================
    # VALIDATE ACTIONS
    # =====================================================

    if not isinstance(
        actions,
        list
    ):

        raise ValueError(
            "actions harus berupa list."
        )


    actions = [

        str(action)
        .strip()
        .upper()

        for action in actions
    ]


    # Remove duplicates while preserving order

    actions = list(
        dict.fromkeys(actions)
    )


    if not actions:

        raise ValueError(
            "actions kosong."
        )


    for action in actions:

        if action not in LABELS:

            raise ValueError(
                f"Invalid action label: {action}"
            )


    # =====================================================
    # VALIDATE CONFIDENCE
    # =====================================================

    if not 0 <= confidence <= 1:

        raise ValueError(
            f"Invalid confidence: {confidence}"
        )


    # =====================================================
    # VALIDATE AMBIGUOUS
    # =====================================================

    if not isinstance(
        ambiguous,
        bool
    ):

        raise ValueError(
            "ambiguous harus boolean."
        )


    return {

        "actions":
            actions,

        "confidence":
            confidence,

        "ambiguous":
            ambiguous,
    }


# =========================================================
# STATUS
# =========================================================

print()
print("=" * 70)
print("ACTION CLASSIFIER — TEACHER API")
print("=" * 70)

print(
    "Model          :",
    MODEL
)

print(
    "API key        :",
    "OK"
)

print(
    "Labels         :",
    len(LABELS)
)

print(
    "call_teacher() :",
    "OK"
)

print(
    "parser         :",
    "OK"
)

print("=" * 70)


ACTION CLASSIFIER — TEACHER API
Model          : deepseek-v4-flash
API key        : OK
Labels         : 8
call_teacher() : OK
parser         : OK


In [5]:
# =========================================================
# CELL 5 — TEACHER API SANITY TEST
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

SANITY_CASES = [
    {
        "command": "cat /etc/os-release",
        "expected": ["READ"],
    },
    {
        "command": "ls -la /tmp",
        "expected": ["READ"],
    },
    {
        "command": "echo hello > output.txt",
        "expected": ["WRITE"],
    },
    {
        "command": "cp source.txt backup.txt",
        "expected": ["READ", "WRITE"],
    },
    {
        "command": "rm test.txt",
        "expected": ["DELETE"],
    },
    {
        "command": "python app.py",
        "expected": ["EXECUTE"],
    },
    {
        "command": "curl https://example.com/file -o file",
        "expected": ["NETWORK", "WRITE"],
    },
    {
        "command": "wget https://example.com/archive.tar.gz",
        "expected": ["NETWORK", "WRITE"],
    },
    {
        "command": "sudo apt update",
        "expected": [
            "NETWORK",
            "PRIVILEGED",
            "SYSTEM_CHANGE",
        ],
    },
    {
        "command": "sudo apt install nginx",
        "expected": [
            "NETWORK",
            "INSTALL",
            "PRIVILEGED",
            "SYSTEM_CHANGE",
        ],
    },
    {
        "command": "systemctl status ssh",
        "expected": ["READ"],
    },
    {
        "command": "sudo systemctl restart nginx",
        "expected": [
            "EXECUTE",
            "PRIVILEGED",
            "SYSTEM_CHANGE",
        ],
    },
]


def normalize_label_set(labels):

    return set(
        str(label).strip().upper()
        for label in labels
    )


results = []


print()
print("=" * 90)
print("ACTION CLASSIFIER — TEACHER SANITY TEST")
print("=" * 90)


for index, case in enumerate(
    SANITY_CASES,
    start=1
):

    command = case["command"]
    expected = case["expected"]

    try:

        raw_output, latency = call_teacher(
            command
        )

        parsed = parse_teacher_output(
            raw_output
        )

        predicted = parsed["actions"]

        exact_match = (
            normalize_label_set(predicted)
            ==
            normalize_label_set(expected)
        )

        row = {
            "index": index,
            "command": command,
            "expected": expected,
            "predicted": predicted,
            "confidence": parsed["confidence"],
            "ambiguous": parsed["ambiguous"],
            "latency": latency,
            "exact_match": exact_match,
            "error": None,
        }

    except Exception as e:

        row = {
            "index": index,
            "command": command,
            "expected": expected,
            "predicted": None,
            "confidence": None,
            "ambiguous": None,
            "latency": None,
            "exact_match": False,
            "error": str(e),
        }


    results.append(row)


    print()
    print(
        f"[{index}/{len(SANITY_CASES)}]",
        command
    )

    print(
        "Expected :",
        expected
    )

    print(
        "Predicted:",
        row["predicted"]
    )

    print(
        "Confidence:",
        row["confidence"]
    )

    print(
        "Match:",
        row["exact_match"]
    )


SANITY_DF = pd.DataFrame(
    results
)


print()
print("=" * 90)

successful = (
    SANITY_DF["error"].isna().sum()
)

exact = (
    SANITY_DF["exact_match"].sum()
)

print(
    "Successful calls :",
    successful,
    "/",
    len(SANITY_DF)
)

print(
    "Exact matches    :",
    exact,
    "/",
    len(SANITY_DF)
)

print(
    "Exact accuracy   :",
    round(
        exact / len(SANITY_DF),
        4
    )
)

print("=" * 90)


ACTION CLASSIFIER — TEACHER SANITY TEST

[1/12] cat /etc/os-release
Expected : ['READ']
Predicted: ['READ']
Confidence: 1.0
Match: True

[2/12] ls -la /tmp
Expected : ['READ']
Predicted: ['READ']
Confidence: 1.0
Match: True

[3/12] echo hello > output.txt
Expected : ['WRITE']
Predicted: ['WRITE']
Confidence: 1.0
Match: True

[4/12] cp source.txt backup.txt
Expected : ['READ', 'WRITE']
Predicted: ['READ', 'WRITE']
Confidence: 1.0
Match: True

[5/12] rm test.txt
Expected : ['DELETE']
Predicted: ['DELETE']
Confidence: 1.0
Match: True

[6/12] python app.py
Expected : ['EXECUTE']
Predicted: ['EXECUTE']
Confidence: 1.0
Match: True

[7/12] curl https://example.com/file -o file
Expected : ['NETWORK', 'WRITE']
Predicted: ['NETWORK', 'WRITE']
Confidence: 1.0
Match: True

[8/12] wget https://example.com/archive.tar.gz
Expected : ['NETWORK', 'WRITE']
Predicted: ['NETWORK', 'WRITE']
Confidence: 1.0
Match: True

[9/12] sudo apt update
Expected : ['NETWORK', 'PRIVILEGED', 'SYSTEM_CHANGE']
Predicted:

In [6]:
# =========================================================
# CELL 6 — CREATE MANUAL SEED DATASET
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

MANUAL_SAMPLES = [

    # =====================================================
    # READ
    # =====================================================

    {
        "command": "cat /etc/os-release",
        "description": "Read operating system release information",
        "actions": ["READ"],
    },

    {
        "command": "ls -la /tmp",
        "description": "List files in temporary directory",
        "actions": ["READ"],
    },

    {
        "command": "pwd",
        "description": "Print current working directory",
        "actions": ["READ"],
    },

    {
        "command": "whoami",
        "description": "Display current user",
        "actions": ["READ"],
    },

    {
        "command": "systemctl status ssh",
        "description": "Inspect SSH service status",
        "actions": ["READ"],
    },


    # =====================================================
    # WRITE
    # =====================================================

    {
        "command": "echo hello > output.txt",
        "description": "Write text into a file",
        "actions": ["WRITE"],
    },

    {
        "command": "touch test.txt",
        "description": "Create an empty file",
        "actions": ["WRITE"],
    },

    {
        "command": "cp source.txt backup.txt",
        "description": "Read source file and create a copy",
        "actions": ["READ", "WRITE"],
    },


    # =====================================================
    # DELETE
    # =====================================================

    {
        "command": "rm test.txt",
        "description": "Delete a file",
        "actions": ["DELETE"],
    },

    {
        "command": "rm -r temp_folder",
        "description": "Delete a directory recursively",
        "actions": ["DELETE"],
    },


    # =====================================================
    # EXECUTE
    # =====================================================

    {
        "command": "python app.py",
        "description": "Execute a Python program",
        "actions": ["EXECUTE"],
    },

    {
        "command": "bash script.sh",
        "description": "Execute a shell script",
        "actions": ["EXECUTE"],
    },


    # =====================================================
    # NETWORK
    # =====================================================

    {
        "command": "curl https://example.com",
        "description": "Request data from a remote HTTP server",
        "actions": ["NETWORK"],
    },

    {
        "command": "ping example.com",
        "description": "Send network echo requests to remote host",
        "actions": ["NETWORK"],
    },

    {
        "command": "curl https://example.com/file -o file",
        "description": "Download a remote file to local storage",
        "actions": ["NETWORK", "WRITE"],
    },


    # =====================================================
    # INSTALL
    # =====================================================

    {
        "command": "pip install requests",
        "description": "Install a Python package",
        "actions": [
            "NETWORK",
            "INSTALL",
            "SYSTEM_CHANGE",
        ],
    },

    {
        "command": "npm install express",
        "description": "Install a Node.js dependency",
        "actions": [
            "NETWORK",
            "INSTALL",
            "SYSTEM_CHANGE",
        ],
    },


    # =====================================================
    # PRIVILEGED + SYSTEM_CHANGE
    # =====================================================

    {
        "command": "sudo apt update",
        "description": "Update system package repository metadata",
        "actions": [
            "NETWORK",
            "PRIVILEGED",
            "SYSTEM_CHANGE",
        ],
    },

    {
        "command": "sudo apt install nginx",
        "description": "Install nginx using system package manager",
        "actions": [
            "NETWORK",
            "INSTALL",
            "PRIVILEGED",
            "SYSTEM_CHANGE",
        ],
    },

    {
        "command": "chmod 600 config.txt",
        "description": "Change file permissions",
        "actions": [
            "SYSTEM_CHANGE",
        ],
    },

    {
        "command": "sudo systemctl restart nginx",
        "description": "Restart nginx system service",
        "actions": [
            "EXECUTE",
            "PRIVILEGED",
            "SYSTEM_CHANGE",
        ],
    },


    # =====================================================
    # MULTI-ACTION / CHAINED
    # =====================================================

    {
        "command": "cat input.txt | curl -X POST https://example.com -d @-",
        "description": "Read local data and send it to a remote server",
        "actions": [
            "READ",
            "NETWORK",
        ],
    },

    {
        "command": "wget https://example.com/app.py -O app.py && python app.py",
        "description": "Download a Python program and execute it",
        "actions": [
            "NETWORK",
            "WRITE",
            "EXECUTE",
        ],
    },
]


manual_df = pd.DataFrame(
    MANUAL_SAMPLES
)


manual_df["confidence"] = 1.0

manual_df["ambiguous"] = False

manual_df["source"] = (
    "manual_seed"
)

manual_df["split_origin"] = (
    "manual"
)


print()
print("=" * 70)
print("ACTION CLASSIFIER — MANUAL SEED")
print("=" * 70)

print(
    "Samples:",
    len(manual_df)
)

print()

display(
    manual_df[
        [
            "command",
            "actions"
        ]
    ]
)

print("=" * 70)


ACTION CLASSIFIER — MANUAL SEED
Samples: 23



,command,actions
0,cat /etc/os-release,[READ]
1,ls -la /tmp,[READ]
2,pwd,[READ]
3,whoami,[READ]
4,systemctl status ssh,[READ]
5,echo hello > output.txt,[WRITE]
6,touch test.txt,[WRITE]
7,cp source.txt backup.txt,"[READ, WRITE]"
8,rm test.txt,[DELETE]
9,rm -r temp_folder,[DELETE]


In [7]:
# =========================================================
# CELL 7 — MANUAL SEED VALIDATION
# MODEL 01 — ACTION CLASSIFIER
# =========================================================


# =========================================================
# 1. BASIC VALIDATION
# =========================================================

validation_errors = []


for index, row in manual_df.iterrows():

    command = str(
        row["command"]
    ).strip()

    actions = row[
        "actions"
    ]


    if not command:

        validation_errors.append(
            {
                "row": index,
                "error": "empty_command",
            }
        )


    if not validate_actions(
        actions
    ):

        validation_errors.append(
            {
                "row": index,
                "error": (
                    f"invalid_actions: {actions}"
                ),
            }
        )


# =========================================================
# 2. DUPLICATE COMMAND CHECK
# =========================================================

duplicate_mask = (
    manual_df["command"]
    .str.strip()
    .str.lower()
    .duplicated(
        keep=False
    )
)


duplicate_rows = (
    manual_df[
        duplicate_mask
    ]
)


# =========================================================
# 3. LABEL COVERAGE
# =========================================================

label_counts = {
    label: 0
    for label in LABELS
}


for actions in manual_df["actions"]:

    for label in actions:

        label_counts[label] += 1


label_distribution_df = pd.DataFrame(
    [
        {
            "label": label,
            "count": count,
        }

        for label, count
        in label_counts.items()
    ]
)


# =========================================================
# 4. STATUS
# =========================================================

print()
print("=" * 70)
print("ACTION CLASSIFIER — MANUAL VALIDATION")
print("=" * 70)


print(
    "Validation errors :",
    len(validation_errors)
)

print(
    "Duplicate commands:",
    len(duplicate_rows)
)


print()
print("LABEL COVERAGE")
print("-" * 70)

display(
    label_distribution_df
)


if validation_errors:

    print()
    print("ERRORS:")

    display(
        pd.DataFrame(
            validation_errors
        )
    )


if len(duplicate_rows) > 0:

    print()
    print("DUPLICATES:")

    display(
        duplicate_rows[
            [
                "command",
                "actions"
            ]
        ]
    )


assert (
    len(validation_errors) == 0
), "Manual dataset memiliki validation error."


assert (
    len(duplicate_rows) == 0
), "Manual dataset memiliki duplicate command."


missing_labels = [

    label

    for label, count
    in label_counts.items()

    if count == 0
]


assert (
    not missing_labels
), (
    "Label belum ter-cover: "
    + str(missing_labels)
)


print()
print(
    "Manual seed validation: PASS"
)

print("=" * 70)


ACTION CLASSIFIER — MANUAL VALIDATION
Validation errors : 0
Duplicate commands: 0

LABEL COVERAGE
----------------------------------------------------------------------


,label,count
0,READ,7
1,WRITE,5
2,DELETE,2
3,EXECUTE,4
4,NETWORK,9
5,INSTALL,3
6,PRIVILEGED,3
7,SYSTEM_CHANGE,6



Manual seed validation: PASS


In [8]:
# =========================================================
# CELL 8 — SAVE MANUAL SEED + SANITY REPORT
# MODEL 01 — ACTION CLASSIFIER
# =========================================================


# =========================================================
# 1. SAVE MANUAL DATASET
# =========================================================

manual_df.to_json(
    MANUAL_SEED_PATH,
    orient="records",
    lines=True,
    force_ascii=False
)


# =========================================================
# 2. SAVE LABEL DISTRIBUTION
# =========================================================

label_distribution_df.to_csv(
    LABEL_DISTRIBUTION_PATH,
    index=False
)


# =========================================================
# 3. SAVE TEACHER SANITY RESULTS
# =========================================================

TEACHER_SANITY_PATH = (
    f"{REPORT_DIR}/teacher_sanity_test.csv"
)


SANITY_SAVE_DF = (
    SANITY_DF.copy()
)


SANITY_SAVE_DF[
    "expected"
] = SANITY_SAVE_DF[
    "expected"
].apply(
    json.dumps
)


SANITY_SAVE_DF[
    "predicted"
] = SANITY_SAVE_DF[
    "predicted"
].apply(
    lambda x:
        json.dumps(x)
        if isinstance(x, list)
        else x
)


SANITY_SAVE_DF.to_csv(
    TEACHER_SANITY_PATH,
    index=False
)


# =========================================================
# 4. SUMMARY
# =========================================================

teacher_exact_accuracy = (
    SANITY_DF[
        "exact_match"
    ].mean()
)


teacher_avg_confidence = (
    SANITY_DF[
        "confidence"
    ].dropna().mean()
)


teacher_avg_latency = (
    SANITY_DF[
        "latency"
    ].dropna().mean()
)


SUMMARY = {

    "manual_samples":
        int(
            len(manual_df)
        ),

    "teacher_sanity_samples":
        int(
            len(SANITY_DF)
        ),

    "teacher_exact_accuracy":
        float(
            teacher_exact_accuracy
        ),

    "teacher_avg_confidence":
        float(
            teacher_avg_confidence
        ),

    "teacher_avg_latency_sec":
        float(
            teacher_avg_latency
        ),
}


TEACHER_SANITY_SUMMARY_PATH = (
    f"{REPORT_DIR}/teacher_sanity_summary.json"
)


with open(
    TEACHER_SANITY_SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        SUMMARY,
        f,
        indent=2,
        ensure_ascii=False
    )


# =========================================================
# 5. STATUS
# =========================================================

print()
print("=" * 70)
print("ACTION CLASSIFIER — SEED DATA SAVED")
print("=" * 70)

print(
    "Manual seed :",
    MANUAL_SEED_PATH
)

print(
    "Label report:",
    LABEL_DISTRIBUTION_PATH
)

print(
    "Teacher test:",
    TEACHER_SANITY_PATH
)

print(
    "Summary     :",
    TEACHER_SANITY_SUMMARY_PATH
)

print()
print(
    "Manual samples          :",
    SUMMARY[
        "manual_samples"
    ]
)

print(
    "Teacher exact accuracy  :",
    round(
        SUMMARY[
            "teacher_exact_accuracy"
        ],
        4
    )
)

print(
    "Teacher avg confidence  :",
    round(
        SUMMARY[
            "teacher_avg_confidence"
        ],
        4
    )
)

print(
    "Teacher avg latency (s) :",
    round(
        SUMMARY[
            "teacher_avg_latency_sec"
        ],
        3
    )
)

print("=" * 70)


ACTION CLASSIFIER — SEED DATA SAVED
Manual seed : /content/drive/MyDrive/action_classifier/raw/manual_seed_v1.jsonl
Label report: /content/drive/MyDrive/action_classifier/reports/label_distribution.csv
Teacher test: /content/drive/MyDrive/action_classifier/reports/teacher_sanity_test.csv
Summary     : /content/drive/MyDrive/action_classifier/reports/teacher_sanity_summary.json

Manual samples          : 23
Teacher exact accuracy  : 1.0
Teacher avg confidence  : 1.0
Teacher avg latency (s) : 1.009


In [9]:
# =========================================================
# CELL 9 — PUBLIC DATASET SOURCE SETUP
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

PUBLIC_DATASET_NAME = (
    "liontech/NL2Bash"
)

PUBLIC_DATASET_CONFIG = (
    "train"
)

MAX_PUBLIC_SAMPLES = 40000


print()
print("=" * 70)
print("ACTION CLASSIFIER — PUBLIC DATASET SETUP")
print("=" * 70)

print(
    "Dataset source :",
    PUBLIC_DATASET_NAME
)

print(
    "Dataset config :",
    PUBLIC_DATASET_CONFIG
)

print(
    "Max samples    :",
    MAX_PUBLIC_SAMPLES
)

print("=" * 70)


ACTION CLASSIFIER — PUBLIC DATASET SETUP
Dataset source : liontech/NL2Bash
Dataset config : train
Max samples    : 40000


In [10]:
# =========================================================
# CELL 9.1 — DOWNLOAD PUBLIC DATASET
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

from datasets import load_dataset

import os


# =========================================================
# CACHE DIRECTORY
# =========================================================

HF_CACHE_DIR = os.path.join(
    RAW_DIR,
    "hf_cache"
)

os.makedirs(
    HF_CACHE_DIR,
    exist_ok=True
)


# =========================================================
# DOWNLOAD DATASET
# =========================================================

print()
print("=" * 70)
print("ACTION CLASSIFIER — PUBLIC DATASET DOWNLOAD")
print("=" * 70)

print(
    "Dataset:",
    PUBLIC_DATASET_NAME
)

print()


dataset = load_dataset(
    PUBLIC_DATASET_NAME,
    PUBLIC_DATASET_CONFIG,
    cache_dir=HF_CACHE_DIR,
)


# =========================================================
# STATUS
# =========================================================

print()
print("Available splits:")

for split_name in dataset.keys():

    print(
        f"{split_name:12}:",
        len(dataset[split_name])
    )


print()
print(
    "Cache:",
    HF_CACHE_DIR
)

print("=" * 70)


ACTION CLASSIFIER — PUBLIC DATASET DOWNLOAD
Dataset: liontech/NL2Bash



README.md:   0%|          | 0.00/700 [00:00<?, ?B/s]


Available splits:
train       : 40639

Cache: /content/drive/MyDrive/action_classifier/raw/hf_cache


In [11]:
# =========================================================
# CELL 10 — INSPECT PUBLIC DATASET
# MODEL 01 — ACTION CLASSIFIER
# =========================================================


# =========================================================
# SELECT TRAIN SPLIT
# =========================================================

if "train" not in dataset:

    raise RuntimeError(
        "Train split tidak ditemukan."
    )


public_split = dataset[
    "train"
]


# =========================================================
# OPTIONAL LIMIT
# =========================================================

if (
    MAX_PUBLIC_SAMPLES
    and len(public_split) > MAX_PUBLIC_SAMPLES
):

    public_split = (
        public_split
        .shuffle(
            seed=SEED
        )
        .select(
            range(
                MAX_PUBLIC_SAMPLES
            )
        )
    )


# =========================================================
# STATUS
# =========================================================

print()
print("=" * 70)
print("ACTION CLASSIFIER — PUBLIC DATASET INSPECTION")
print("=" * 70)

print(
    "Samples:",
    len(public_split)
)

print(
    "Columns:",
    public_split.column_names
)

print()

print(
    "Features:"
)

print(
    public_split.features
)

print()


for i in range(
    min(
        5,
        len(public_split)
    )
):

    print(
        f"[{i}]",
        public_split[i]
    )

    print()


print("=" * 70)


ACTION CLASSIFIER — PUBLIC DATASET INSPECTION
Samples: 40000
Columns: ['nl', 'bash']

Features:
{'nl': Value('string'), 'bash': Value('string')}

[0] {'nl': 'Convert input_file from one encoding to another and output to stdout', 'bash': 'iconv -f from_encoding -t to_encoding input_file'}

[1] {'nl': 'Find all files with the extension ".txt" in the current directory and its subdirectories up to 3 levels deep, print the results, and replace any numbers in the filenames with a space followed by the number, then sort the results numerically by the number.', 'bash': '`find / -maxdepth 3 -name "*.txt" -print | sed \'s/\\(\\(.*\\)\\([[:digit:]]\\)\\)/\\1 \\3/g\' |sort -n -k 2`'}

[2] {'nl': 'Print the 9th field of all lines beginning with a hyphen (-) in the output of the ls -Rl command.', 'bash': "ls -Rl | awk '/^-/{print $9}'"}

[3] {'nl': 'Find all files with the extension ".js" in the current directory and up to 4 levels of subdirectories, delete them, and then remove all blank lines fro

In [14]:
# =========================================================
# CELL 10 — LOAD + INSPECT PUBLIC DATASET
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

print()
print("=" * 70)
print("LOADING PUBLIC DATASET")
print("=" * 70)

dataset = load_dataset(
    PUBLIC_DATASET_NAME,
    "train"
)

print()
print("Available splits:")

for split_name in dataset.keys():
    print(
        "-",
        split_name,
        ":",
        len(dataset[split_name])
    )


# =========================================================
# SELECT TRAIN SPLIT
# =========================================================

if "train" not in dataset:
    raise RuntimeError(
        "Train split tidak ditemukan."
    )

public_split = dataset["train"]


# =========================================================
# OPTIONAL LIMIT
# =========================================================

if (
    MAX_PUBLIC_SAMPLES
    and len(public_split) > MAX_PUBLIC_SAMPLES
):
    public_split = (
        public_split
        .shuffle(seed=SEED)
        .select(
            range(MAX_PUBLIC_SAMPLES)
        )
    )


# =========================================================
# INSPECTION
# =========================================================

print()
print("=" * 70)
print("ACTION CLASSIFIER — PUBLIC DATASET INSPECTION")
print("=" * 70)

print("Samples:", len(public_split))
print("Columns:", public_split.column_names)

print()
print("Features:")
print(public_split.features)

print()
print("Example records:")

for i in range(
    min(5, len(public_split))
):
    print()
    print(
        f"[{i}]",
        public_split[i]
    )

print()
print("=" * 70)


LOADING PUBLIC DATASET


ValueError: Config name is missing.
Please pick one among the available configs: ['train', 'test']
Example of usage:
	`load_dataset('liontech/NL2Bash', 'train')`

In [ ]:
# =========================================================
# CELL 11 — NORMALIZE PUBLIC DATASET
# MODEL 01 — ACTION CLASSIFIER
# =========================================================


# =========================================================
# 1. COLUMN CANDIDATES
# =========================================================

COMMAND_CANDIDATES = [
    "command",
    "cmd",
    "bash",
    "shell",
    "target",
    "output",
]

DESCRIPTION_CANDIDATES = [
    "description",
    "nl",
    "text",
    "question",
    "instruction",
    "intent",
    "input",
]


available_columns = (
    public_split.column_names
)


print()
print("=" * 70)
print("PUBLIC DATASET — COLUMN DETECTION")
print("=" * 70)

print(
    "Available columns:",
    available_columns
)


# =========================================================
# 2. AUTO-DETECT
# =========================================================

command_column = None

for candidate in COMMAND_CANDIDATES:

    if candidate in available_columns:

        command_column = candidate
        break


description_column = None

for candidate in DESCRIPTION_CANDIDATES:

    if candidate in available_columns:

        description_column = candidate
        break


print()
print(
    "Command column    :",
    command_column
)

print(
    "Description column:",
    description_column
)


if command_column is None:

    raise RuntimeError(
        "Command column tidak berhasil dideteksi. "
        f"Columns: {available_columns}"
    )


# =========================================================
# 3. NORMALIZATION FUNCTIONS
# =========================================================

def normalize_command(text):

    if text is None:
        return ""

    text = str(text)

    text = text.strip()

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text


def normalize_description(text):

    if text is None:
        return ""

    text = str(text)

    text = text.strip()

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text


# =========================================================
# 4. BUILD DATAFRAME
# =========================================================

records = []


for row in public_split:

    command = normalize_command(
        row.get(
            command_column,
            ""
        )
    )

    if description_column:

        description = normalize_description(
            row.get(
                description_column,
                ""
            )
        )

    else:

        description = ""


    if not command:

        continue


    records.append(
        {
            "command": command,
            "description": description,
            "source": PUBLIC_DATASET_NAME,
            "split_origin": "public_seed",
        }
    )


public_df = pd.DataFrame(
    records
)


print()
print(
    "Normalized samples:",
    len(public_df)
)


print()
display(
    public_df.head(10)
)

print("=" * 70)

In [ ]:
# =========================================================
# CELL 12 — EXACT DEDUP + PRELIMINARY DATASET
# MODEL 01 — ACTION CLASSIFIER
# =========================================================


# =========================================================
# 1. PRE-DEDUP STATS
# =========================================================

before_count = len(
    public_df
)


# =========================================================
# 2. DEDUP KEY
# =========================================================

public_df[
    "command_key"
] = (
    public_df["command"]
    .str.strip()
    .str.lower()
)


# =========================================================
# 3. EXACT DUPLICATE CHECK
# =========================================================

duplicate_mask = (
    public_df[
        "command_key"
    ]
    .duplicated(
        keep="first"
    )
)


duplicate_count = int(
    duplicate_mask.sum()
)


duplicates_df = (
    public_df[
        duplicate_mask
    ]
    .copy()
)


dedup_df = (
    public_df[
        ~duplicate_mask
    ]
    .copy()
)


dedup_df = (
    dedup_df
    .drop(
        columns=[
            "command_key"
        ]
    )
    .reset_index(
        drop=True
    )
)


after_count = len(
    dedup_df
)


# =========================================================
# 4. SAVE
# =========================================================

dedup_df.to_json(
    PUBLIC_RAW_PATH,
    orient="records",
    lines=True,
    force_ascii=False
)


dedup_df.to_json(
    DEDUP_DATASET_PATH,
    orient="records",
    lines=True,
    force_ascii=False
)


PUBLIC_DUPLICATES_PATH = (
    f"{REPORT_DIR}/public_exact_duplicates.csv"
)


duplicates_df.to_csv(
    PUBLIC_DUPLICATES_PATH,
    index=False
)


# =========================================================
# 5. BASIC QUALITY STATS
# =========================================================

command_lengths = (
    dedup_df[
        "command"
    ]
    .str.len()
)


description_missing = int(
    (
        dedup_df[
            "description"
        ].str.len()
        == 0
    ).sum()
)


print()
print("=" * 70)
print("ACTION CLASSIFIER — PUBLIC DATASET PRELIMINARY")
print("=" * 70)

print(
    "Before dedup       :",
    before_count
)

print(
    "Exact duplicates   :",
    duplicate_count
)

print(
    "After dedup        :",
    after_count
)

print(
    "Missing description:",
    description_missing
)

print(
    "Avg command length :",
    round(
        command_lengths.mean(),
        2
    )
)

print(
    "Max command length :",
    int(
        command_lengths.max()
    )
)

print()
print(
    "Saved public seed  :",
    PUBLIC_RAW_PATH
)

print(
    "Saved dedup seed   :",
    DEDUP_DATASET_PATH
)

print(
    "Duplicate report   :",
    PUBLIC_DUPLICATES_PATH
)

print("=" * 70)